# 02 - AF3 setup for UPOs with heme and Mg

## Introduction

This notebook shows a practical workflow to run AlphaFold 3 (AF3) jobs for UPO sequences with one heme cofactor (`HEM`) and one magnesium ion (`MG`) using `prepare_proteins`.

The tutorial covers:

- Preparing the AF3 input JSON files with `setUpAlphaFold3`
- Submitting the generated commands
- Collecting and exporting top models with `collectAlphaFold3Results`
- Applying the built-in strict PROT-HEME-MG filter


## 1. Import modules

As in the existing project tutorials, we use both `prepare_proteins` and `bsc_calculations`.


In [ ]:
import json
import os
import shutil
import subprocess

import bsc_calculations
import pandas as pd
import prepare_proteins


## 2. Prepare a FASTA file with UPO sequences
Use your real UPO sequences in production. The short sequences below are placeholders so the setup code can run.


In [ ]:
fasta_path = "upo_sequences.fasta"

if not os.path.exists(fasta_path):
    with open(fasta_path, "w") as handle:
        handle.write(
            ">UPO_001\n"
            "MALVAVLLAASAFAAPVAAEGVTVVGPDATVKPASPVTVTTVGATPTVTAVDGYTVRV\n"
            ">UPO_002\n"
            "MKTLTALALALAGSAFAAPAQATVVGPKATVKPSAPVTVTTVGATPTVTVVDGYTVRV\n"
        )

print("FASTA file:", fasta_path)

## 3. Initialize `sequenceModels`
The FASTA headers become model names in the generated AF3 jobs.


In [ ]:
sequences = prepare_proteins.sequenceModels(fasta_path)
print("Loaded models:", sequences.sequences_names)

## 4. Optional example: model a dimeric protein
Many enzymes are active as dimers, so it is useful to know how to encode chain stoichiometry explicitly.

How AF3 input represents a dimer:
- Each chain is a separate protein entry with its own chain ID.
- Homodimer: chains `A` and `B` share the same sequence.
- Heterodimer: chains `A` and `B` use different sequences.

In this example we build a homodimer from one loaded UPO sequence and request two HEM + two MG (one pair per protomer).
We also use `unpack_model_seeds=True` so each seed is an independent parallel job.


In [ ]:
# Build a homodimer from the first loaded model sequence
base_model = sequences.sequences_names[0]
base_sequence = sequences.sequences[base_model]

if not isinstance(base_sequence, str):
    raise ValueError("This tutorial dimer example expects FASTA monomer entries as strings.")

dimer_name = f"{base_model}_HOMODIMER"
dimer_sequences = {
    dimer_name: {
        "A": base_sequence,
        "B": base_sequence,
    }
}
# Heterodimer template:
# dimer_sequences = {"MODEL_AB": {"A": sequence_A, "B": sequence_B}}

dimer_models = prepare_proteins.sequenceModels(dimer_sequences)

dimer_jobs = dimer_models.setUpAlphaFold3(
    job_folder="af3_upo_dimer_example",
    ligands={"HEM": 2, "MG": 2},
    model_seeds=[1, 2, 3],
    unpack_model_seeds=True,
    skip_finished=True,
)

print(f"Prepared {len(dimer_jobs)} jobs for dimer model: {dimer_name}")
if dimer_jobs:
    print("First dimer job prepared. Submit using bsc_calculations.mn5.jobArrays(..., program=\'alphafold3\').")


## 5. Generate AF3 jobs for protein + HEM + MG
Important: keep ligand order as `HEM`, then `MG` when you plan to use the built-in strict filter later.

The strict filter assumes chain indices:

- 0 -> protein
- 1 -> heme
- 2 -> magnesium

We also define seed settings here because they control reproducibility and how many independent model attempts you compare.


In [ ]:
job_folder = "af3_upo_heme_mg"

# Each seed is an independent AF3 attempt for the same input system.
# Compare several seeds to avoid over-interpreting one lucky run.
model_seeds = [1, 2, 3]

# False: one AF3 job per model, all seeds inside modelSeeds=[...]
# True: one AF3 job per seed (seed_1/, seed_2/, ...) for easy parallelisation
unpack_model_seeds = False

jobs = sequences.setUpAlphaFold3(
    job_folder=job_folder,
    ligands=["HEM", "MG"],
    model_seeds=model_seeds,
    unpack_model_seeds=unpack_model_seeds,
    skip_finished=True,
)

print(f"Prepared {len(jobs)} jobs for {len(sequences.sequences_names)} models")
if unpack_model_seeds:
    print("Mode: one job per seed (parallel-friendly seed folders)")
else:
    print("Mode: one job per model (all seeds grouped in one AF3 input)")

if jobs:
    print("First job prepared. Submit using bsc_calculations.mn5.jobArrays(..., program=\'alphafold3\').")


## 6. Seed strategy: why this matters scientifically
AF3 is not fully deterministic: changing the random seed can lead to different candidate structures.

Interpretation for modeling:
- One seed = one independent hypothesis for the same molecular system.
- Multiple seeds = a small ensemble, useful to judge robustness.

Practical options in `setUpAlphaFold3`:
- `unpack_model_seeds=False` (default): one job per model, JSON includes all seeds.
- `unpack_model_seeds=True`: one job per seed, easier to parallelize and monitor per-seed completion.

For teaching and first analyses, using 3-5 seeds per model is usually a good compromise between runtime and confidence.


## 7. Create a SLURM array script with `bsc_calculations`
Following the style of tutorial 01, we build a SLURM array script with `bsc_calculations.mn5.jobArrays`.

For AF3 on MN5, `program="alphafold3"` enables AF3-specific setup in the helper.

Useful flags in this call:

- `partition`: queue (`acc_bscls`, `acc_debug`, `gp_bscls`, `gp_debug`)
- `gpus`: number of GPUs requested
- `jobs_range` / `group_jobs_by`: split large command lists into manageable arrays


In [ ]:
slurm_script = "slurm_array_af3_upo.sh"
slurm_job_name = "AF3_UPO_HEM_MG"

# Recommended MN5 partition for AF3
partition = "acc_bscls"

bsc_calculations.mn5.jobArrays(
    jobs,
    script_name=slurm_script,
    job_name=slurm_job_name,
    output=slurm_job_name,
    partition=partition,
    program="alphafold3",
    gpus=1,
)

print(f"Generated {slurm_script}")
print(f"Submit with: sbatch {slurm_script}")

launch_slurm = False
if launch_slurm:
    subprocess.run(["sbatch", slurm_script], check=True)
else:
    print("Dry run only. Set launch_slurm=True to submit.")


## 8. Sanity-check outputs before scoring
Before collecting scores, verify that each expected model produced at least one `ranking_scores.csv`.

Why this check is important:
- It prevents silent bias from analyzing only successful jobs.
- It tells you early which models still need re-submission.
- If you used `unpack_model_seeds=True`, `ranking_files_found` is also a quick proxy for how many seed jobs finished per model.


In [ ]:
sanity_rows = []

for model_name in sequences.sequences_names:
    model_root = os.path.join(job_folder, model_name)
    ranking_paths = []

    if os.path.isdir(model_root):
        for root, _, files in os.walk(model_root):
            if "ranking_scores.csv" in files:
                ranking_paths.append(os.path.join(root, "ranking_scores.csv"))

    sanity_rows.append({
        "model": model_name,
        "ranking_files_found": len(ranking_paths),
        "finished": len(ranking_paths) > 0,
    })

sanity_df = pd.DataFrame(sanity_rows).sort_values("model").reset_index(drop=True)
print(sanity_df)

missing_models = sanity_df.loc[~sanity_df["finished"], "model"].tolist()
print(f"\nFinished models: {int(sanity_df['finished'].sum())}/{len(sanity_df)}")

if missing_models:
    print("Models still missing completed AF3 outputs:")
    for model_name in missing_models:
        print(f" - {model_name}")
else:
    print("All models have at least one ranking_scores.csv file.")


## 9. Collect AF3 ranking scores and export top models
After jobs finish, collect metrics and export top predictions as PDB files.

Here we set `append_model_index` automatically from `top_models` so filenames stay unique whenever more than one model is exported.


In [ ]:
top_models = 5
append_model_index = top_models > 1

scores_df, selected_df, copied_paths = sequences.collectAlphaFold3Results(
    af_folder=job_folder,
    output_folder="af3_upo_heme_mg_top_models",
    metric="ranking_score",
    top_models=top_models,
    append_model_index=append_model_index,
    return_selected=True,
)

print(f"top_models={top_models}, append_model_index={append_model_index}")
print(scores_df.head())
print(selected_df.head())
print(copied_paths)


## 10. Apply strict PROT-HEME-MG filtering (optional)
`filter_strict=True` adds model-level pass/fail annotations with reasons.


In [ ]:
top_models_filtered = 5
append_model_index_filtered = top_models_filtered > 1

filtered_scores, filtered_selected, filtered_copied, filter_info = sequences.collectAlphaFold3Results(
    af_folder=job_folder,
    output_folder="af3_upo_heme_mg_filtered_models",
    metric="ranking_score",
    top_models=top_models_filtered,
    append_model_index=append_model_index_filtered,
    return_selected=True,
    filter_strict=True,
    return_filter=True,
)

print(f"top_models_filtered={top_models_filtered}, append_model_index_filtered={append_model_index_filtered}")
print(filter_info["counts"])
print(filter_info["reasons"])


In [ ]:
if "filtered_scores" in globals():
    cols = [
        "ranking_score",
        "summary_iptm",
        "prot_ptm",
        "pair_iptm_prot_heme",
        "pair_iptm_prot_mg",
        "pae_min_prot_heme",
        "pae_min_prot_mg",
        "af3_strict_pass",
        "af3_strict_reason",
    ]
    cols = [c for c in cols if c in filtered_scores.columns]
    print(filtered_scores[cols].head())

## 11. Troubleshooting checklist
- If no `ranking_scores.csv` files exist, AF3 likely failed before finishing.
- If `metric` is not found, inspect available columns in the generated ranking CSV.
- If copied structures are missing, check `overwrite`, `top_models`, and whether `append_model_index` is set from `top_models > 1`.
- If strict filtering gives unexpected results, verify ligand order is still protein -> heme -> magnesium.
- If jobs fail immediately, validate `bsc_alphafold` and `sbatch` in your active environment.
- If SLURM submission fails, open the generated `slurm_array_af3_upo.sh` and verify partition/account values for your project.
